<a href="https://colab.research.google.com/github/shbhm96/GEN-AI-_-GAN-AND-Convulational-2-D-model/blob/main/GEN_AI___GAN_AND_Convulational_2_D_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')


#load Libraries

from keras.datasets import mnist
import keras
from keras.layers import *
from keras.models import Sequential, Model
from keras.optimizers import Adam
from keras.layers import LeakyReLU

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
(x_train,_),(_,_)=mnist.load_data()

In [ ]:
x_train.shape

(60000, 28, 28)

In [ ]:
x_train= (x_train-127.5)/127.5

print(x_train.min())
print(x_train.max())

-1.0
1.0


In [ ]:
total_Epochs = 50
Batch_size = 256
Half_batch = Batch_size/2

No_of_Batches = int(x_train.shape[0]/Batch_size)

Noise_DIM = 100

adam = Adam(learning_rate = 0.0002, beta_1 = 0.5)

In [ ]:
#Generate the MOdel : Upsampling

generator = Sequential()

generator.add(Dense(units=7*7*128, input_shape = (Noise_DIM,)))
generator.add(Reshape((7,7,128)))
generator.add(LeakyReLU(alpha=0.2))
generator.add(BatchNormalization())

#(7,7,128) > (14,14,64)

generator.add(Conv2DTranspose(filters=64, kernel_size=(3,3), strides=(2,2), padding='same'))
generator.add(LeakyReLU(alpha = 0.2))
generator.add(BatchNormalization())

#(14,14,64) > (28,28,1)

generator.add(Conv2DTranspose(filters=1, kernel_size=(3,3), strides=(2,2), padding='same'))
generator.add(Activation('tanh'))

generator.compile(loss=keras.losses.binary_crossentropy, optimizer=adam)


generator.summary()




Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 6272)           │       633,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_5 (Reshape)             │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 14, 14, 64)     │        73,792 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_3              │ (None, 28, 28, 1)      │           577 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 28, 28, 1)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 708,609 (2.70 MB)

 Trainable params: 708,225 (2.70 MB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
#down Sampling
#(28,28,1) -> (14,14,64)

descriminator = Sequential()
descriminator.add(Conv2D(filters=64, kernel_size=(3,3), strides=(2,2), padding='same', input_shape=(28,28,1)))
descriminator.add(LeakyReLU(alpha=0.2))

#(28,28,1) -> (7,7,128)
descriminator.add(Conv2D(filters=128, kernel_size=(3,3), strides=(2,2), padding='same'))
descriminator.add(LeakyReLU(alpha=0.2))

#(7,7,128) -> 6272
descriminator.add(Flatten())
descriminator.add(Dense(100))
descriminator.add(LeakyReLU(alpha=0.2))

descriminator.add(Dense(1))
descriminator.add(Activation('sigmoid'))

descriminator.compile(loss=keras.losses.binary_crossentropy, optimizer=adam)

descriminator.summary()


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_8 (LeakyReLU)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_9 (LeakyReLU)       │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 100)            │       627,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_10 (LeakyReLU)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │           101 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 701,897 (2.68 MB)

 Trainable params: 701,897 (2.68 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#combine MOdel

descriminator.trainable = False

gen_input =  Input(shape =(Noise_DIM,))

generated_img = generator(gen_input)

gan_output = descriminator(generated_img)

#funstional API
model = Model(gen_input, gan_output)
model.compile(loss=keras.losses.binary_crossentropy, optimizer=adam)
model.summary()

Model: "functional_43"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_9 (Sequential)       │ (None, 28, 28, 1)      │       708,609 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_14 (Sequential)      │ (None, 1)              │       701,897 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,410,506 (5.38 MB)

 Trainable params: 708,225 (2.70 MB)

 Non-trainable params: 702,281 (2.68 MB)

In [1]:
x_train = x_train.reshape(-1,28,28,1)

NameError: name 'x_train' is not defined

In [ ]:
x_train.shape

NameError: name 'x_train' is not defined

In [ ]:
#traininf Loop

#descriminator loss
d_loss = []

#generator_loss
g_loss = []

for epoch in range(total_Epochs):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0

    #mini batch gradient decent
    for step in range(No_of_Batches):
        #train

        #step 1 train descriminator


        #get real data
        idx = np.random.randint(0,60000,Half_batch)
        real_imgs = x_train[idx]

        #get fake data
        noise = np.random.normal(0,1,size = (Half_batch,Noise_DIM))
        fake_imgs = generator.predict(noise)

        #labels
        real_y = np.ones((Half_batch,1))*0.9
        fake_y = np.zeros((Half_batch,1))

        #train
        d_loss_real = descriminator.train_on_batch(real_imgs, real_y)
        d_loss_fake = descriminator.train_on_batch(fake_imgs, fake_y)

        d_loss = 0.5 * d_loss_real + 0.5 * d_loss_fake

        epoch_d_loss += d_loss

        #step 2 train generator
        noise = np.random.normal(0,1,size=(Batch_size, Noise_DIM))

        ground_truth_y =np.ones((Batch_size,1))

        epoch_g_loss += epoch_g_loss

        #=====================
        print("epoch", epoch+1)
        print("Disc Loss", epoch_d_loss/No_of_Batches)
        print("Disc Loss", epoch_g_loss/No_of_Batches)


        d_loss.append(epoch_d_loss/No_of_Batches)
        g_loss.append(epoch_g_loss/No_of_Batches)

        if(epoch+1)%10 == 0:
          generator.save("Generator.h5")
          display_images()


NameError: name 'total_Epochs' is not defined